# Lecture 7. Multiple regression and checking the model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture07/lecture07.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: Newbold, Carlson and Thorne, sections 12.1 to 12.7 and 13.4 to
13.6.

Any data give a least-squares line. In Lecture 6 we fitted one for spend on years as a customer, and we stated four
assumptions behind its test and interval without checking them. Today we follow a routine that you can use for every
regression, also in the exam assignment:

| Step | What we do | Part |
|---|---|---|
| Fit | The line from Lecture 6: spend on tenure | |
| Check | Is the line what is happening? Residual plot and tests | 1 |
| Fix | Robust standard errors, or the log of spend | 2 |
| Extend | Add age, then visits | 3 |
| Check again | Check the final model, then predict and report | 4 |

Where Python does something differently from Newbold, a section "... in Python and in Newbold" says so. Each part
ends with an exercise: you predict a result, run the cell, change one value, and answer a question. You do not need
to write any code. The last section holds a summary table, the symbols and the Danish terms.

Run the setup cell first. It loads the RetailCorp customer data from the course repository.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
customers = pd.read_csv(url + "customers_sample.csv")   # 1,000 customers drawn at random, as in Lecture 6

print(f"Customers: {len(customers):,}")
customers.head()

## The case and the question

The same 1,000 RetailCorp customers as in Lecture 6. Today we use these columns:

| Column | Content |
|---|---|
| `spend_2025` | Total spend in DKK in 2025 (y) |
| `tenure_years` | Years since the customer's first order, on 1 January 2025 |
| `age` | Age on 1 January 2025 |
| `visits_2025` | Logged-in visits to the shop in 2025 |

**Lead question.** Do customers who have been with RetailCorp longer spend more, compared with customers of the same
age, and how far can we trust the numbers?

## Part 1. Checking the line (Check)

**The line from Lecture 6.** The line is estimated by least squares, as in Lecture 6. Nothing about the estimate
changes today. The checks ask whether the line describes the data, and whether its standard errors can be trusted.

In [ ]:
model = smf.ols("spend_2025 ~ tenure_years", data=customers).fit()
print(model.params)

**The four assumptions and how we check them.** The errors ε<sub>i</sub> cannot be observed. The residuals
e<sub>i</sub> = y<sub>i</sub> − ŷ<sub>i</sub> are our view of them, so every check uses the residuals.

| In plain words | In Newbold | How we check it | Checked by |
|---|---|---|---|
| The average of y is a straight line in x | Linear model | Residual plot: no curve | Eye |
| Customers are independent of each other | Errors uncorrelated | How the data were collected | The design |
| The scatter around the line is the same for all customers | Errors with constant variance | Residual plot: no fan. Breusch-Pagan test. | Eye and a test |
| The errors are normally distributed | Normal errors | Histogram of the residuals; Skew, Kurtosis and Jarque-Bera in the output | Eye and a test |

**How to read a residual plot.** Across: the fitted value ŷ<sub>i</sub>, the line's prediction for customer i. Up:
the residual e<sub>i</sub>, how far the customer lies above or below the line. A residual of 0 means the customer
lies exactly on the line, so the dashed line at 0 stands for the regression line itself. A good line leaves a shapeless cloud around it. A curve means the average is not a straight
line in x. A fan means the scatter grows with the fitted value.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(model.fittedvalues, model.resid, s=8, alpha=0.5)
plt.axhline(0, color="darkorange", linestyle="--")
plt.xlabel("Fitted value: the line's spend for the customer, DKK")
plt.ylabel("Residual, DKK")
plt.show()

**Our residual plot.** No curve: the dots are centered on 0 from left to right, so the straight line is a fair
summary. But there is a fan: the scatter grows from left to right. Is the fan strong enough to matter, or could it be
chance in this sample? A test answers that.

### A test for the fan

If the scatter is the same everywhere, the squared residuals do not grow with the fitted values. So we regress
e<sup>2</sup> on ŷ (an extra regression, the auxiliary regression) and see how much of e<sup>2</sup> it accounts for:

- H<sub>0</sub>: the same variance for all customers. H<sub>1</sub>: the variance depends on ŷ<sub>i</sub>.
- Auxiliary regression: e<sub>i</sub><sup>2</sup> = a<sub>0</sub> + a<sub>1</sub>ŷ<sub>i</sub>.
- Test statistic: n R<sup>2</sup><sub>aux</sub>, which under H<sub>0</sub> follows approximately a χ<sup>2</sup>
  distribution with 1 degree of freedom in large samples (Newbold 13.6).
  One degree of freedom, because the auxiliary regression has one explanatory variable, ŷ, and we test one slope.

The cell does the test by the formula and then with statsmodels' `het_breuschpagan`.

In [ ]:
aux = pd.DataFrame({"e2": model.resid**2, "yhat": model.fittedvalues})
r2_aux = smf.ols("e2 ~ yhat", data=aux).fit().rsquared
nr2 = len(aux) * r2_aux                          # n R²
p = stats.chi2.sf(nr2, df=1)
print(f"R2 of the auxiliary regression = {r2_aux:.5f}")
print(f"nR2 = {nr2:.2f}, p = {p:.1e}")
print(f"critical value = {stats.chi2.ppf(0.95, df=1):.3f}")

lm, lm_p, f, f_p = het_breuschpagan(model.resid, model.model.exog)
print(f"LM = {lm:.2f}, p = {lm_p:.1e}")

**What it means for the line.** The errors are larger for customers with a higher predicted spend. The slope of about
200 DKK per year is still our estimate, but the standard error and the interval from Lecture 6 assume equal scatter,
and here they understate the uncertainty.

### Spread tests in Python and in Newbold

| Point | Newbold 13.6 | `het_breuschpagan` in Python |
|---|---|---|
| Regress e<sup>2</sup> on | The fitted values ŷ | All the explanatory variables in the model |
| Degrees of freedom | 1 | The number of explanatory variables in the model |
| What you get | One number to compare with χ<sup>2</sup> | Four numbers: LM, its p-value, an F version and its p-value. Use the first two. |

With one x the two give the same number, because ŷ is a straight line in x. With several x they differ a little
(Part 4). Either version is fine in your notebook; say which one you used.

### Are the residuals normal?

A histogram of the residuals, and three numbers that `summary()` prints at the bottom: `Skew` (0 for a normal
distribution), `Kurtosis` (3 for a normal distribution) and `Jarque-Bera`, a test of normality built from the two.
`Omnibus` is another such test.

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.hist(model.resid, bins=np.arange(-4000, 10501, 500))
plt.xlabel("Residual, DKK")
plt.show()

jb, jb_p = stats.jarque_bera(model.resid)
print(f"Skew = {stats.skew(model.resid):.2f}, Kurtosis = {stats.kurtosis(model.resid, fisher=False):.2f}, "
      f"Jarque-Bera = {jb:.1f}, p = {jb_p:.2f}")

In [ ]:
print(model.summary())    # Skew, Kurtosis and Jarque-Bera are in the bottom block

The residuals are skewed to the right with long tails: a few customers lie far above the line. With 1,000 customers
the t test for the slope is still close to right, as for x̄ in Lecture 1. With a small sample it may not be.

### The verdict for the Lecture 6 line

| Assumption | What we see | What it means | Next |
|---|---|---|---|
| Straight line | No curve in the residual plot | The line is a fair summary of the average | Nothing |
| Independent customers | A random sample of customers | No problem from the design | Nothing |
| The same scatter for all | A fan, and Breusch-Pagan rejects (p = 2.0e-06) | The slope is still centered on the population value, but here its standard error, t, p and interval are too optimistic | Part 2: robust standard errors, or ln spend |
| Normal errors | Skewed residuals (Skew 1.95) | With 1,000 customers the tests are still close to right; with a small sample they may not be | Part 2: ln spend |

Residuals could be related when rows are in time order (Durbin-Watson in the output checks that), or when several
rows come from the same household or store. Ours are a random sample of customers, so we do not test it.

### Exercise 1. The checks for age

**Predict.** The cell fits spend against age and runs the checks. Will the residual plot show a curve? A fan?

**Run** the cell. It draws the residual plot and prints Breusch-Pagan, its p-value, and the Skew and Kurtosis of the
residuals.

In [ ]:
x_var = "age"      # change to "tenure_years" in step 3

m1 = smf.ols(f"spend_2025 ~ {x_var}", data=customers).fit()
plt.figure(figsize=(7, 4))
plt.scatter(m1.fittedvalues, m1.resid, s=8, alpha=0.5)
plt.axhline(0, color="darkorange", linestyle="--")
plt.xlabel(f"Fitted value (spend against {x_var}), DKK")
plt.ylabel("Residual, DKK")
plt.show()

lm, lm_p, f, f_p = het_breuschpagan(m1.resid, m1.model.exog)
print(f"x = {x_var}: Breusch-Pagan = {lm:.2f}, p = {lm_p:.1e}, "
      f"Skew = {stats.skew(m1.resid):.2f}, Kurtosis = {stats.kurtosis(m1.resid, fisher=False):.2f}")

**Change** `x_var = "age"` to `"tenure_years"` and run the cell again.

**Answer.** Which of the four assumptions hold for the age model, and which fail? What do the two models have in
common?

## Part 2. Two responses (Fix)

When the scatter is not the same for all customers, there are two responses:

1. **Keep the model, fix the standard errors.** Robust standard errors drop the assumption of equal scatter. The line
   and its coefficients stay the same; the standard errors, t, p and the interval change.
2. **Change the model.** Model the log of spend instead of spend. If the error is a percentage of the level, it
   becomes a fixed distance in logs, so the scatter in logs can be the same for all customers.

The two can be combined: a log model with robust standard errors (Part 4).

### Robust standard errors

`fit(cov_type="HC1")` computes the standard error from each customer's own squared residual, so it does not assume
equal scatter. They are also called heteroscedasticity-robust or White standard errors.

In [ ]:
model_r = smf.ols("spend_2025 ~ tenure_years", data=customers).fit(cov_type="HC1")

lo, hi = model.conf_int().loc["tenure_years"]
lo_r, hi_r = model_r.conf_int().loc["tenure_years"]
print(f"{'':16}{'classical':>14}{'robust (HC1)':>16}")
print(f"{'b1':16}{model.params['tenure_years']:14.3f}{model_r.params['tenure_years']:16.3f}")
print(f"{'standard error':16}{model.bse['tenure_years']:14.3f}{model_r.bse['tenure_years']:16.3f}")
print(f"{'t (z with HC1)':16}{model.tvalues['tenure_years']:14.2f}{model_r.tvalues['tenure_years']:16.2f}")
print(f"{'95% interval':16}{lo:7.1f} to {hi:5.1f}{lo_r:9.1f} to {hi_r:5.1f}")
print(f"{'F-statistic':16}{model.fvalue:14.2f}{model_r.fvalue:16.2f}")

### Demonstration: which standard error is right?

The cell draws 1,000 samples of 1,000 customers from all 60,000 and fits the line in each. The slopes vary from
sample to sample; their standard deviation is what a standard error should measure. The cell compares it with the
average classical and robust standard errors, and counts how often the 95% intervals contain the population slope.
It takes a few seconds.

In [ ]:
population = pd.read_csv(url + "customers_population.csv")   # all 60,000 customers
true_slope = smf.ols("spend_2025 ~ tenure_years", data=population).fit().params["tenure_years"]

rng = np.random.default_rng(7)
slopes, se_classical, se_robust = [], [], []
for i in range(1000):
    idx = rng.choice(len(population), 1000, replace=False)
    sample = population.iloc[idx]
    fit = smf.ols("spend_2025 ~ tenure_years", data=sample).fit()
    slopes.append(fit.params["tenure_years"])
    se_classical.append(fit.bse["tenure_years"])
    se_robust.append(fit.get_robustcov_results(cov_type="HC1").bse[1])

slopes, se_classical, se_robust = np.array(slopes), np.array(se_classical), np.array(se_robust)
t_crit = stats.t.ppf(0.975, df=998)
print(f"Population slope: {true_slope:.1f}")
print(f"Standard deviation of the 1,000 slopes: {slopes.std(ddof=1):.1f}")
print(f"Average classical standard error: {se_classical.mean():.1f}")
print(f"Average robust standard error: {se_robust.mean():.1f}")
print(f"Intervals containing the population slope: classical {np.mean(abs(slopes - true_slope) <= t_crit * se_classical):.1%}, "
      f"robust {np.mean(abs(slopes - true_slope) <= t_crit * se_robust):.1%}")

plt.figure(figsize=(7, 3.5))
plt.hist(slopes, bins=np.arange(130, 281, 6), density=True, alpha=0.6)
grid = np.linspace(130, 280, 300)
plt.plot(grid, stats.norm.pdf(grid, slopes.mean(), se_classical.mean()), "--", label="classical standard error")
plt.plot(grid, stats.norm.pdf(grid, slopes.mean(), se_robust.mean()), label="robust standard error")
plt.xlabel("Slope in each of 1,000 samples (DKK per year)")
plt.legend()
plt.show()

### Robust standard errors in Python and in Newbold

- Newbold 13.6 says the usual tests are not valid under unequal scatter and suggests logs; it has no robust standard
  errors. They are in Wooldridge, chapter 8. Use either response, and say why.
- The default is `cov_type="nonrobust"`, shown as `Covariance Type` in the output, with note [1] under it. Look at
  `Covariance Type` in every output.
- HC0 to HC3 differ in small-sample corrections. We use HC1, which adds a degrees-of-freedom correction,
  n/(n − K − 1), where K is the number of explanatory variables.
- With a robust `cov_type` the table shows `z` and `P>|z|`, and the interval uses 1.96. `use_t=True` gives t. With
  1,000 customers the difference does not matter.
- The F-statistic becomes a robust test too.

In [ ]:
print("HC1:", round(model_r.bse["tenure_years"], 3))
print("HC3:", round(smf.ols("spend_2025 ~ tenure_years", data=customers).fit(cov_type="HC3").bse["tenure_years"], 3))

### The log of spend

Money amounts are often skewed: many small amounts, a few large ones. If the error is a percentage of the level, it
is larger in DKK for customers who spend more: a fan. In logs, a percentage becomes a fixed distance (Newbold 12.7).
We use the natural logarithm, ln, which is `np.log` in Python.

In [ ]:
customers["log_spend"] = np.log(customers["spend_2025"])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(customers["spend_2025"], bins=np.arange(0, 12501, 500))
axes[0].set_xlabel("spend_2025, DKK")
axes[1].hist(customers["log_spend"], bins=np.arange(5.0, 9.61, 0.2), color="darkorange")
axes[1].set_xlabel("ln spend_2025")
plt.show()
print(f"Skew of spend: {stats.skew(customers['spend_2025']):.2f}, skew of ln spend: {stats.skew(customers['log_spend']):.2f}")

**The log model.** ln y<sub>i</sub> = β<sub>0</sub> + β<sub>1</sub>x<sub>i</sub> + ε<sub>i</sub>. A one-unit
difference in x goes with about 100 × b<sub>1</sub> percent difference in y, exactly 100 × (exp(b<sub>1</sub>) − 1)
percent. The approximation is close when b<sub>1</sub> is small, below about 0.1.

In [ ]:
model_log = smf.ols("log_spend ~ tenure_years", data=customers).fit()
b = model_log.params["tenure_years"]
print(model_log.params)
print(f"About {100 * b:.1f}% per year, exactly {100 * (np.exp(b) - 1):.2f}%")

**For the manager.** Customers one year apart in tenure differ in spend by about 9%, at every level of tenure. In
Lecture 6 it was the same number of DKK at every level.

### Checking the log model

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(model_log.fittedvalues, model_log.resid, s=8, alpha=0.5)
plt.axhline(0, color="darkorange", linestyle="--")
plt.xlabel("Fitted value: the line's ln spend")
plt.ylabel("Residual (ln)")
plt.show()

for name, m in [("Levels", model), ("Logs", model_log)]:
    lm, lm_p, f, f_p = het_breuschpagan(m.resid, m.model.exog)
    p_text = f"{lm_p:.1e}" if lm_p < 0.001 else f"{lm_p:.2f}"
    print(f"{name}: Breusch-Pagan = {lm:.2f}, p = {p_text}, Skew = {stats.skew(m.resid):.2f}, "
          f"Kurtosis = {stats.kurtosis(m.resid, fisher=False):.2f}, R2 = {m.rsquared:.3f}")

No fan, and the test does not reject equal scatter (p = 0.60). The residuals are close to symmetric, with kurtosis
close to 3.

**Levels against logs.** R² cannot be compared between the two models. R² is the share of the variation in y, and y
differs: DKK in one model, ln DKK in the other. Compare the two models with the residual plots and the tests, and by
which reading of the slope fits the question.

### Three forms of the line

| Model | Estimated line | Reading of b<sub>1</sub> | Example |
|---|---|---|---|
| y on x | ŷ = b<sub>0</sub> + b<sub>1</sub>x | b<sub>1</sub> units of y more per unit of x | 201.8 DKK per year of tenure |
| ln y on x | predicted ln y = b<sub>0</sub> + b<sub>1</sub>x | about 100 × b<sub>1</sub> percent more y per unit of x | 0.0900: about 9.0%, exactly 9.4% per year of tenure |
| ln y on ln x | predicted ln y = b<sub>0</sub> + b<sub>1</sub> ln x | about b<sub>1</sub> percent more y per 1 percent more x: the elasticity | Exercise 2: visits |

Newbold 12.7 presents the log-log model and calls b<sub>1</sub> an elasticity, as in demand models: 1% higher price,
b<sub>1</sub>% change in quantity.

### Logs in Python and in Newbold

- `np.log` is the natural log, ln. `np.log10` is base 10 and gives other coefficients. Use `np.log` for the percent
  readings.
- `smf.ols("np.log(spend_2025) ~ tenure_years", ...)` works too, with a long name in the output. We make a new
  column instead.
- `np.log(0)` gives −inf and a negative value gives nan, with only a warning. Check the smallest value first. Here
  spend and visits are all above 0.
- `np.exp` undoes `np.log` (Part 4).

### Exercise 2. Visits in logs

**Predict.** The cell fits ln spend on ln visits. Will b<sub>1</sub> be positive? Will it be above or below 1?

**Run** the cell. It prints b<sub>1</sub>, its t value, R², and the percent difference in spend for `more_visits`
percent more visits.

In [ ]:
more_visits = 10      # percent more visits; change to 100 (twice as many visits) in step 3

customers["log_visits"] = np.log(customers["visits_2025"])
m_e2 = smf.ols("log_spend ~ log_visits", data=customers).fit()
b = m_e2.params["log_visits"]
print(f"b1 = {b:.4f}, t = {m_e2.tvalues['log_visits']:.2f}, R2 = {m_e2.rsquared:.3f}")
print(f"{more_visits}% more visits: about {b * more_visits:.1f}% more spend, "
      f"exactly {100 * ((1 + more_visits / 100)**b - 1):.1f}%")

**Change** `more_visits = 10` to `100` and run the cell again.

**Answer.** How much more do customers with 10% more visits spend, and with twice as many? Why is the second not ten
times the first?

## Part 3. Several explanatory variables (Extend)

Older customers have been customers longer, and older customers spend more. So the simple slope of ln spend on tenure
picks up both the tenure difference and the age difference that comes with it. No residual plot or test in Part 1
shows this, because the checks cannot find a variable that is missing. It takes knowledge of the customers.

In [ ]:
customers[["age", "tenure_years", "log_spend"]].corr().round(3)

**Question.** Among customers of the same age, do those who have been with RetailCorp longer spend more, and how
much?

**What we know.** ln spend on tenure alone: b = 0.0900, about 9.0%, exactly 9.4% per year. Age and tenure are correlated (r =
0.485). n = 1,000, α = 0.05.

**What we want.** The tenure coefficient with age in the model, why it differs from 0.0900, and whether it differs
from 0.

**The model.** ln y<sub>i</sub> = β<sub>0</sub> + β<sub>1</sub>x<sub>1i</sub> + β<sub>2</sub>x<sub>2i</sub> +
ε<sub>i</sub>, with x<sub>1</sub> tenure and x<sub>2</sub> age. β<sub>1</sub> is the difference in average ln
spend per year of tenure, at the same age. Least squares chooses b<sub>0</sub>, b<sub>1</sub> and b<sub>2</sub> as
in Lecture 6; Newbold 12.2 gives the formulas for two x, and Python computes them.

**Why the tenure coefficient changes.** With b̃<sub>1</sub> the simple slope (age left out) and d the slope of age
on tenure:

b̃<sub>1</sub> = b<sub>1</sub> + b<sub>2</sub> d

The simple slope equals the slope at the same age, plus the part of the age difference that comes with one more year
of tenure. A left-out variable changes the slope when it is related to y (b<sub>2</sub> is not 0) and related to x
(d is not 0). Newbold 13.4 calls this specification bias; it is also called omitted variable bias.

In [ ]:
m2 = smf.ols("log_spend ~ tenure_years + age", data=customers).fit()
print(m2.params)
aux_age = smf.ols("age ~ tenure_years", data=customers).fit()
d = aux_age.params["tenure_years"]
b1, b2 = m2.params["tenure_years"], m2.params["age"]
print(f"{b1 + b2 * d:.4f}")              # the simple slope
print(f"{100 * (np.exp(b1) - 1):.2f}")      # percent

**The t test.** As in Lecture 6, t = b<sub>j</sub> / s<sub>b<sub>j</sub></sub>, now with n − K − 1 degrees of
freedom, where K is the number of explanatory variables (here 2, so 997 degrees of freedom).

In [ ]:
n, K = len(customers), 2
t_tenure = m2.params["tenure_years"] / m2.bse["tenure_years"]
print(f"s_b1 = {m2.bse['tenure_years']:.6f}, t = {t_tenure:.2f}, critical value = {stats.t.ppf(0.975, df=n - K - 1):.4f}")

**For the manager.** Among customers of the same age, those with one more year as customers spend about 6% more (9%
when age is ignored). Part of the tenure difference was an age difference.

**Reading the coefficients.** Tenure 0.0573: customers one year apart in tenure and of the same age differ in spend by
about 5.7%, exactly 5.9%. Age 0.0137: customers one year apart in age and with the same tenure differ by about 1.4%. The intercept
is ln spend at tenure 0 and age 0, an extrapolation, since no customer is younger than 18. "The same age" is a
comparison in the data: the model does not follow customers over time, and differences left out of the model can
still be in b<sub>1</sub>.

### Tests and fit with several x

- R² never falls when an x is added, even a useless one. Adjusted R² can fall:
  R̄² = 1 − (SSE / (n − K − 1)) / (SST / (n − 1)). Use it to compare models with different numbers of x, for the
  same y and the same customers.
- The F test in the output tests whether all slopes are 0 at once: H<sub>0</sub>: β<sub>1</sub> = β<sub>2</sub> = 0,
  F = (SSR / K) / (SSE / (n − K − 1)). With one x it was t².

Watch the letters: statsmodels' `ssr` is Newbold's SSE (the residual sum of squares).

In [ ]:
sse, sst = m2.ssr, m2.centered_tss
adj_r2 = 1 - (sse / (n - K - 1)) / (sst / (n - 1))
f_stat = ((sst - sse) / K) / (sse / (n - K - 1))
print(f"R2 = {m2.rsquared:.4f}, adjusted R2 = {adj_r2:.4f} (statsmodels {m2.rsquared_adj:.4f})")
print(f"F = {f_stat:.2f} (statsmodels {m2.fvalue:.2f}), p = {m2.f_pvalue:.1e}")
print(m2.summary())

### Related explanatory variables

When explanatory variables are closely related, the data cannot separate their coefficients well: the standard errors
grow, while the coefficients are still centered on the population values. The variance inflation factor,
VIF<sub>j</sub> = 1 / (1 − R<sub>j</sub><sup>2</sup>), says how many times larger the variance of b<sub>j</sub> is
because x<sub>j</sub> is related to the other x. R<sub>j</sub><sup>2</sup> comes from regressing x<sub>j</sub> on the
other x; with two x it is r². Values above 10 (some say 5) are a warning.

Newbold 13.5 checks the correlations and regresses each x on the others, which is what the VIF does. `Cond. No.` in
the output is another measure; statsmodels adds a warning when it is large.

In [ ]:
print(f"By hand: 1 / (1 - {aux_age.rsquared:.4f}) = {1 / (1 - aux_age.rsquared):.4f}")

X = m2.model.exog              # column 0 is the intercept
names = m2.model.exog_names
for i in range(1, X.shape[1]):
    print(names[i], round(variance_inflation_factor(X, i), 2))

### The earlier value as a control

Suppose the data also had each customer's spend in 2024, and we added ln spend in 2024 to the model:

ln spend<sub>2025</sub> = β<sub>0</sub> + β<sub>1</sub> tenure + β<sub>2</sub> age + β<sub>3</sub> ln
spend<sub>2024</sub> + ε

- β<sub>1</sub> then compares 2025 spend between customers who had the same spend in 2024. It reads as a difference
  in the change from one year to the next only if β<sub>3</sub> is close to 1.
- Last year's value usually accounts for much of this year's, so R² rises a lot.
- Lasting differences between customers that are not in the data show up in last year's spend, so they are partly
  held fixed.

Our data have no 2024 spend. The exam assignment does this in 5.4 with productivity in 2023, and Lecture 10 practices
it.

### Exercise 3. Adding visits

**Predict.** The cell adds `visits_2025` to the model with tenure and age. Which coefficients will change, and in
which direction?

**Run** the cell. It prints the coefficients and adjusted R² with and without visits.

In [ ]:
base = "tenure_years + age"      # change to "tenure_years" (age left out) in step 3

without = smf.ols(f"log_spend ~ {base}", data=customers).fit()
with_visits = smf.ols(f"log_spend ~ {base} + visits_2025", data=customers).fit()
for name, m in [("without visits", without), ("with visits", with_visits)]:
    coefs = ", ".join(f"{k} {v:.4f}" for k, v in m.params.items() if k != "Intercept")
    print(f"{name:>15}: {coefs}, adjusted R2 {m.rsquared_adj:.3f}")

**Change** `base = "tenure_years + age"` to `"tenure_years"` (age left out) and run the cell again.

**Answer.** Why does the age coefficient go up when visits are added? Why is the visits coefficient smaller when age is
left out?

## Part 4. Check again, predict, report (Check again)

### Checking the final model

The final model is ln spend on tenure, age and visits. We run the same checks as in Part 1, and the VIF.

In [ ]:
m3 = smf.ols("log_spend ~ tenure_years + age + visits_2025", data=customers).fit()

plt.figure(figsize=(7, 4))
plt.scatter(m3.fittedvalues, m3.resid, s=8, alpha=0.5)
plt.axhline(0, color="darkorange", linestyle="--")
plt.xlabel("Fitted value: ln spend from tenure, age and visits")
plt.ylabel("Residual (ln)")
plt.show()

lm, lm_p, f, f_p = het_breuschpagan(m3.resid, m3.model.exog)
aux3 = pd.DataFrame({"e2": m3.resid**2, "yhat": m3.fittedvalues})
nr2_3 = len(aux3) * smf.ols("e2 ~ yhat", data=aux3).fit().rsquared
print(f"Newbold's version: nR2 = {nr2_3:.2f}, 1 df, p = {stats.chi2.sf(nr2_3, df=1):.2f}")
print(f"Breusch-Pagan (statsmodels, 3 df) = {lm:.2f}, p = {lm_p:.2f}, Skew = {stats.skew(m3.resid):.2f}, "
      f"Kurtosis = {stats.kurtosis(m3.resid, fisher=False):.2f}")
X = m3.model.exog
for i in range(1, X.shape[1]):
    print("VIF", m3.model.exog_names[i], round(variance_inflation_factor(X, i), 2))
m3_r = smf.ols("log_spend ~ tenure_years + age + visits_2025", data=customers).fit(cov_type="HC1")
print(f"Standard error of tenure: classical {m3.bse['tenure_years']:.4f}, robust {m3_r.bse['tenure_years']:.4f}")

No curve and no fan; neither version of the test rejects equal scatter (with three x's they give different numbers,
on 1 and 3 degrees of freedom); the residuals are close to normal; the VIFs are small,
so tenure, age and visits can be told apart. Robust standard errors barely change here. We report them anyway, as the
exam assignment does.

### Predicting spend for a customer

A customer with 5 years as a customer, aged 40, with 24 visits. `new` needs a column for every x in the formula, with
the same names. The model predicts ln spend; `np.exp` takes it back to DKK.

In [ ]:
new = pd.DataFrame({"tenure_years": [5], "age": [40], "visits_2025": [24]})
print("Predicted ln spend:", round(m3.predict(new).iloc[0], 4))
print("Predicted spend, DKK:", round(np.exp(m3.predict(new)).iloc[0], 1))

print(f"Whole sample: median {customers['spend_2025'].median():,.0f} DKK, mean {customers['spend_2025'].mean():,.0f} DKK, "
      f"exp of the mean ln spend {np.exp(customers['log_spend'].mean()):,.0f} DKK")

The exponential of a predicted ln spend is a typical spend for such customers, below their average spend, because
spend is skewed. As in Lecture 6, predict only inside the data.

### Comparing models fairly

- **The same customers.** `smf.ols` drops rows with a missing value in any variable of that model. Drop rows with
  a missing value in any of the variables you will use first, for example `df.dropna(subset=[...])`, and check
  `No. Observations`.
- **The same y.** R² and adjusted R² compare models for the same y only (Part 2: levels against logs).
- **Adjusted R² for different numbers of x.** R² rises with every x added; adjusted R² does not have to.
- **The same kind of standard errors.** Say whether they are classical or robust, and use the same in all models you
  compare.

### Exercise 4. Reading the final model

The output below is the final model with robust standard errors (it is also on the slide). With pen and paper:

1. **Read** the tenure coefficient as a percent, about and exactly.
2. **Compute** ln spend for a customer with 5 years, aged 40, with 24 visits, from the printed coefficients, then
   spend in DKK.
3. **Check** which standard errors these are, and where the output says so.
4. **Answer.** Write two sentences on tenure for a manager.

A phone calculator is fine for exp.

In [ ]:
m3_r = smf.ols("log_spend ~ tenure_years + age + visits_2025", data=customers).fit(cov_type="HC1")
print(m3_r.summary())

Run this cell to check your prediction in step 2.

In [ ]:
print(round(np.exp(m3_r.predict(pd.DataFrame({"tenure_years": [5], "age": [40], "visits_2025": [24]}))).iloc[0], 1))

## Reporting a multiple regression

We regressed ln spend in 2025 on years as a customer, age and visits for a random sample of 1,000 customers (OLS with
robust standard errors, HC1, since spend in DKK had unequal scatter). Comparing customers of the same age and with
the same visits, one more year as a customer goes with 4.5% higher spend (95% interval 3.0% to 6.0%; z = 5.94,
p < 0.001). Without age, the tenure coefficient was about 9%. The residuals in logs show no fan and are close
to normal, and the largest VIF is 1.41. The model has R² 0.30 (adjusted R² 0.296) for ln spend. For a
customer with five years, aged 40, with 24 visits it predicts about 1,554 DKK. The data describe customers as they
are.

Report: y, the x's, the method, n, and which standard errors; the coefficient of interest as a percent, with its
interval; what changed when a control was added; the checks in one line, and adjusted R².

## Reference

### Checking and responding in one table

| Assumption | How to check | If it fails | What to do | Python |
|---|---|---|---|---|
| Straight line | Residuals against fitted values | The line misdescribes parts of the data | Logs or another form of x | `model.fittedvalues`, `model.resid` |
| Independent customers | How the data were collected | Standard errors too small | Beyond this course | Durbin-Watson for rows in time order |
| The same scatter | Residual plot; Breusch-Pagan | Standard errors, t, p and intervals wrong | Robust standard errors, or ln y | `het_breuschpagan`; `cov_type="HC1"` |
| Normal errors | Histogram; Skew, Kurtosis, Jarque-Bera | Tests unreliable in small samples | With large n usually nothing; ln y if y is skewed | `summary()` |
| x's not too closely related | VIF; correlations | Large standard errors | Interpret with care, or combine x's | `variance_inflation_factor` |

A left-out variable that is related to both y and x changes the coefficient on x: omitted variable bias. No check in
the output shows it; it takes knowledge of the case.

### Degrees of freedom

| Test | Degrees of freedom | What they count |
|---|---|---|
| t test for one coefficient | n − K − 1 (n − 2 with one x) | Observations left over after estimating the coefficients |
| F test in the output | K and n − K − 1 | The number of slopes tested, and the leftover observations |
| Test for the fan (nR², Newbold) | 1 | The number of slopes tested in the auxiliary regression |
| Breusch-Pagan in statsmodels | K | The number of slopes tested in the auxiliary regression |

### Symbols

| Symbol | Read as | Meaning |
|---|---|---|
| e<sub>i</sub> | e-i | Residual: y<sub>i</sub> − ŷ<sub>i</sub> |
| R<sup>2</sup><sub>aux</sub> | R-squared aux | R² of the auxiliary regression of e² on ŷ |
| χ<sup>2</sup> | chi-squared | The distribution of nR² under H<sub>0</sub> |
| ln y | log y | Natural logarithm of y; `np.log` |
| x<sub>1i</sub>, x<sub>2i</sub> | x-one-i, x-two-i | Customer i's values of the explanatory variables |
| β<sub>1</sub>, β<sub>2</sub> | beta-one, beta-two | Population coefficients: differences at the same value of the other x |
| b̃<sub>1</sub> | b-tilde-one | Slope with the other variable left out |
| K | K | Number of explanatory variables |
| R̄² | R-bar squared | Adjusted R² |
| VIF<sub>j</sub> | VIF-j | Variance inflation factor of x<sub>j</sub> |

### Danish terms

| English | Danish |
|---|---|
| residual plot | residualplot |
| heteroscedasticity, homoscedasticity | heteroskedasticitet, homoskedasticitet |
| auxiliary regression | hjælperegression |
| robust standard errors | robuste standardfejl |
| natural logarithm | naturlig logaritme |
| elasticity | elasticitet |
| multiple regression | multipel regression |
| explanatory variable | forklarende variabel |
| omitted variable bias | bias fra udeladte variable |
| adjusted R² | justeret forklaringsgrad |
| multicollinearity, variance inflation factor | multikollinearitet, variansinflationsfaktor |